Example from Open-Meteo (Dosent Work Well)

In [2]:
pip install openmeteo_requests
pip install requests_cache
pip install retry-requests

   ---------------------------------------- 0.0/731.3 kB ? eta -:--:--
   ---------------------------------------- 731.3/731.3 kB 17.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 12.4 MB/s  0:00:00

   ---------- ----------------------------- 2/8 [qh3]
   ---------- ----------------------------- 2/8 [qh3]
   -------------------- ------------------- 4/8 [jh2]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ------------------------- -------------- 5/8 [urllib3-future]
   ----------------------

In [13]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time  # ← AÑADIR


cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Define locations with their specific features
locations = [
    {
        "name": "Munich",
        "latitude": 48.1374,
        "longitude": 11.5755,
        "hourly": ["shortwave_radiation", "diffuse_radiation", "cloud_cover", "apparent_temperature", "temperature_2m", "relative_humidity_2m", "precipitation"]
    },
    {
        "name": "Stuttgart",
        "latitude": 48.7758,
        "longitude": 9.1829,
        "hourly": ["shortwave_radiation", "diffuse_radiation", "cloud_cover", "apparent_temperature"]
    },
    {
        "name": "Cologne",
        "latitude": 50.9375,
        "longitude": 6.9603,
        "hourly": ["shortwave_radiation", "diffuse_radiation", "cloud_cover", "apparent_temperature", "temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "pressure_msl", "wind_direction_10m", "wind_gusts_10m"]
    },
    {
        "name": "Hanover",
        "latitude": 52.3759,
        "longitude": 9.7320,
        "hourly": ["shortwave_radiation", "diffuse_radiation", "cloud_cover", "apparent_temperature", "wind_speed_10m", "pressure_msl", "wind_direction_10m", "wind_gusts_10m"]
    },
    {
        "name": "Kiel",
        "latitude": 54.3233,
        "longitude": 10.1390,
        "hourly": ["wind_speed_10m", "pressure_msl", "wind_direction_10m", "wind_gusts_10m"]
    },
    {
        "name": "Potsdam",
        "latitude": 52.4009,
        "longitude": 13.0592,
        "hourly": ["wind_speed_10m", "pressure_msl", "wind_direction_10m", "wind_gusts_10m"]
    },
    {
        "name": "Berlin",
        "latitude": 52.5244,
        "longitude": 13.4105,
        "hourly": ["temperature_2m", "apparent_temperature", "shortwave_radiation", "relative_humidity_2m", "precipitation", "is_day"]  # Added is_day here
    },
    {
        "name": "Hamburg",
        "latitude": 53.5507,
        "longitude": 9.9930,
        "hourly": ["temperature_2m", "apparent_temperature", "shortwave_radiation", "relative_humidity_2m", "precipitation"]
    },
    {
        "name": "Frankfurt",
        "latitude": 50.1155,
        "longitude": 8.6842,
        "hourly": ["temperature_2m", "apparent_temperature", "shortwave_radiation", "relative_humidity_2m", "precipitation"]
    },
    
    {
        "name": "Tubingen",
        "latitude": 48.5227,
        "longitude": 9.0522,
        "hourly": [
            "temperature_2m",
            "apparent_temperature",
            "shortwave_radiation",
            "diffuse_radiation",
            "cloud_cover",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "wind_speed_100m",
            "wind_direction_100m",
            "is_day"
        ]
    },
    {
        "name": "Neubrandenburg",
        "latitude": 53.5573,
        "longitude": 13.2610,
        "hourly": [
            "temperature_2m",
            "apparent_temperature",
            "shortwave_radiation",
            "diffuse_radiation",
            "cloud_cover",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "wind_speed_100m",
            "wind_direction_100m",
            "is_day"
        ]
    },
    {
        "name": "Hassel",
        "latitude": 52.8027,
        "longitude": 9.2027,
        "hourly": [
            "temperature_2m",
            "apparent_temperature",
            "shortwave_radiation",
            "diffuse_radiation",
            "cloud_cover",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "wind_speed_100m",
            "wind_direction_100m",
            "is_day"
        ]
    },
    {
        "name": "Bremen",
        "latitude": 53.0758,
        "longitude": 8.8072,
        "hourly": [
            "temperature_2m",
            "apparent_temperature",
            "shortwave_radiation",
            "diffuse_radiation",
            "cloud_cover",
            "relative_humidity_2m",
            "precipitation",
            "pressure_msl",
            "wind_speed_10m",
            "wind_direction_10m",
            "wind_gusts_10m",
            "wind_speed_100m",
            "wind_direction_100m",
            "is_day"
        ]
    }
]

# Define common parameters enter the timeframe also
common_params = {
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "timeformat": "unixtime"
}

# Init Empty
hourly_dfs = []

for loc in locations:
    print(f"Fetching data for {loc['name']}...") #troubleshoot step

    # Update parameters for this location
    params = {
        "latitude": loc["latitude"],
        "longitude": loc["longitude"],
        "hourly": loc["hourly"],
        **common_params
    }

    try:
        responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
        response = responses[0]

        # Process hourly data
        hourly = response.Hourly()
        hourly_data = {"date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        )}

        # Map Open-Meteo variables
        feature_mapping = {
            "shortwave_radiation": "radiation_global",
            "diffuse_radiation": "radiation_diffuse",
            "cloud_cover": "cloud_cover",
            "apparent_temperature": "feels_like",
            "temperature_2m": "temperature",
            "relative_humidity_2m": "humidity",
            "precipitation": "precipitation",
            "wind_speed_10m": "wind_speed_10m",
            "pressure_msl": "air_pressure",
            "wind_direction_10m": "wind_direction_10m",
            "wind_gusts_10m": "wind_gust_10m",
            "wind_speed_100m": "wind_speed_100m",
            "wind_direction_100m": "wind_direction_100m",
            "is_day": "is_day"  # only
        }

        for i, var in enumerate(loc["hourly"]):
            feature_name = feature_mapping.get(var, var)
            hourly_data[f"{feature_name}_{loc['name']}"] = hourly.Variables(i).ValuesAsNumpy()

        df = pd.DataFrame(data=hourly_data)
        hourly_dfs.append(df)

    except Exception as e:
        print(f"Location Data Missing  {loc['name']}: {str(e)}") #check lat-long.
        continue
    finally:
        time.sleep(1.5)  # ← PAUSA entre cada ciudad, dentro del loop

# Merge all dataframes outer
if hourly_dfs:
    merged_df = hourly_dfs[0]
    for df in hourly_dfs[1:]:
        merged_df = pd.merge(merged_df, df, on="date", how="outer")  # Outer join recommended how="inner")

    # Sort by date
    merged_df = merged_df.sort_values("date")

    # Print and export
    print(merged_df.head())
    merged_df.to_csv("Data_weather_2025.csv", index=False)
else:
    print("R// Error ")

Fetching data for Munich...
Fetching data for Stuttgart...
Fetching data for Cologne...
Fetching data for Hanover...
Fetching data for Kiel...
Fetching data for Potsdam...
Fetching data for Berlin...
Fetching data for Hamburg...
Fetching data for Frankfurt...
Fetching data for Tubingen...
Fetching data for Neubrandenburg...
Fetching data for Hassel...
Fetching data for Bremen...
                       date  radiation_global_Munich  \
0 2022-01-01 00:00:00+00:00                      0.0   
1 2022-01-01 01:00:00+00:00                      0.0   
2 2022-01-01 02:00:00+00:00                      0.0   
3 2022-01-01 03:00:00+00:00                      0.0   
4 2022-01-01 04:00:00+00:00                      0.0   

   radiation_diffuse_Munich  cloud_cover_Munich  feels_like_Munich  \
0                       0.0                24.0           4.760664   
1                       0.0                68.0           4.634410   
2                       0.0                63.0           4.344419   
3